In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import os
import re

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_monorail_csv(csv_paths):
    """
    Load one or more Monorail CSV files and apply basic cleaning.
    
    Parameters:
    - csv_paths: str or list of CSV file paths
    
    Returns:
    - df: Combined DataFrame with all samples
    """
    if isinstance(csv_paths, str):
        csv_paths = [csv_paths]
    
    dfs = []
    for fp in csv_paths:
        df = pd.read_csv(fp)
        
        # # Apply filters
        # df = df[(df["Non_Standard_Braking"] == 0) & (df["BC_BadStart"]==0)].copy()
        
        # Extract source from filename
        m = re.search(r"Dati(\d+)", os.path.basename(fp))
        df["Source"] = int(m.group(1)) if m else -1
        
        # Convert "xx sec" strings to float
        for col in df.select_dtypes(include="object").columns:
            if col != "Malfunction":  # Skip Malfunction column
                try:
                    df[col] = df[col].str.replace(" sec", "", regex=False).astype(float)
                except Exception:
                    pass
        
        dfs.append(df)
    
    df_combined = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df_combined)} samples from {len(csv_paths)} file(s)")
    
    return df_combined

In [ ]:
def add_wv_bin(df):
    """
    Add WV_bin column based on WV_MeanPressure.
    
    Bins:
    - 0: WV < 2
    - 1: 2 <= WV <= 3
    - 2: WV > 3
    
    Parameters:
    - df: DataFrame with WV_MeanPressure column
    
    Returns:
    - df: DataFrame with WV_bin column added
    """
    df = df.copy()
    
    if "WV_MeanPressure" in df.columns:
        df["WV_bin"] = df["WV_MeanPressure"].apply(
            lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
        )
    else:
        df["WV_bin"] = np.nan
        print("Warning: WV_MeanPressure column not found, WV_bin set to NaN")
    
    return df

In [ ]:
def remove_label_columns(df):
    """
    Remove label columns that should not be present during inference.
    
    Parameters:
    - df: DataFrame
    
    Returns:
    - df: DataFrame with label columns removed
    """
    df = df.copy()
    
    label_cols = ["LeakageLabel", "label", "Malfunction"]
    cols_to_drop = [c for c in label_cols if c in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"Removed label columns: {cols_to_drop}")
    
    return df

In [ ]:
def get_datetime_column(df):
    """
    Detect a column that contains Time or Datetime in the DataFrame.
    Prefers columns named 'Time' or 'Datetime'; otherwise uses first column
    whose name contains 'time' or 'date' (case-insensitive) and parses as datetime.
    """
    # Prefer exact names
    for name in ["Time", "Datetime", "DateTime", "Date", "Timestamp"]:
        if name in df.columns:
            return name
    # Else first column with "time" or "date" in name that parses as datetime
    for col in df.columns:
        if "time" in col.lower() or "date" in col.lower():
            try:
                sample = df[col].dropna()
                if len(sample) == 0:
                    continue
                parsed = pd.to_datetime(sample.head(20), errors="coerce")
                if parsed.notna().any():
                    return col
            except Exception:
                continue
    return None

In [ ]:
def load_and_prepare_monorail(monorail_paths):
    """
    Complete pipeline: load, clean, and prepare Monorail data for prediction.
    
    Parameters:
    - monorail_paths: str or list of CSV file paths
    - features: list of feature names used during training
    - imputer: fitted imputer from training
    - scaler: fitted scaler from training
    - wv_bin: WV_bin value to filter by (default: 1)
    
    Returns:
    - X_scaled: preprocessed data ready for prediction
    - df_filtered: filtered DataFrame with original indices
    - df_full: full DataFrame with all samples (for reference)
    """
    # Load data
    df_full = load_monorail_csv(monorail_paths)
    mapping = {
    1: "T3000",
    6: "T3000",
    27: "T3000",
    30: "T3000",
    10: "4909",
    11: "4909",
    18: "4909",
    24: "4909",
    5: "4575",
    32: "4575"
}
    df_full["WagonType"] = df_full["Source"].map(mapping)
    # Add WV_bin
    df_full = add_wv_bin(df_full)
    
    # Remove label columns
    df_full = remove_label_columns(df_full)

    return df_full

# Start - model loading

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Path handling (your existing logic is correct)
# ------------------------------------------------------------------
def get_base_dir():
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

PROJECT_DIR = get_base_dir()
MODEL_DIR = PROJECT_DIR / "model"

# Find all joblib files
MODEL_PATHS = sorted(MODEL_DIR.glob("*_inference.joblib"))

print(f"Found {len(MODEL_PATHS)} model(s) in {MODEL_DIR}")


# Test data loading

In [ ]:
test_data_path = [
    "TestBrakefinal_data_raw_Dati30.csv",
    "TestBrakefinal_data_raw_Dati05.csv",
    "TestBrakefinal_data_raw_Dati10.csv",
    "TestBrakefinal_data_raw_Dati11.csv",
    "TestBrakefinal_data_raw_Dati18.csv",
    "TestBrakefinal_data_raw_Dati24.csv",
]

df_test = load_and_prepare_monorail(test_data_path)

df_wv1 = df_test.loc[
    (df_test["Non_Standard_Braking"].eq(0)) &
    (df_test["BC_BadStart"].eq(0)) &
    (df_test["WV_bin"].eq(1))
].copy()

print(f"Filtered test data to {len(df_wv1)} WV_bin=1 healthy samples")


In [ ]:
all_results = {}

for model_path in MODEL_PATHS:
    model_name = model_path.stem
    print(f"\nRunning inference with: {model_name}")

    bundle = joblib.load(model_path)
    pipe = bundle["pipeline"]
    features = bundle["features"]
    
    # Feature alignment (critical safety check)
    X_new = df_wv1[features]

    y_pred = pipe.predict(X_new)
    y_score = pipe.predict_proba(X_new)[:, 1]

    # Store results in a COPY
    df_out = df_wv1.copy()
    df_out["Prediction"] = y_pred
    df_out["Score"] = y_score
    df_out["Model"] = model_name

    all_results[model_name] = df_out


In [ ]:
# Build output.csv: rows where Prediction is True; columns: Datetime, Model, WagonType, Source, Score
# Filter out rows where Datetime is before 2025
dt_col = get_datetime_column(df_wv1)
output_rows = []

for model_name, df_out in all_results.items():
    true_mask = (df_out["Prediction"] == 1) | (df_out["Prediction"] is True)
    df_true = df_out.loc[true_mask].copy()
    if len(df_true) == 0:
        continue
    if dt_col:
        df_true["_dt"] = pd.to_datetime(df_true[dt_col], errors="coerce")
        df_true = df_true.dropna(subset=["_dt"])
        # Filter out rows before 2025
        df_true = df_true[df_true["_dt"].dt.year >= 2025]
        if len(df_true) == 0:
            continue
        df_true["Datetime"] = df_true["_dt"].astype(str)
        for _, row in df_true.iterrows():
            output_rows.append({
                "Datetime": row["Datetime"],
                "Model": model_name,
                "WagonType": row.get("WagonType", ""),
                "Source": row.get("Source", ""),
                "Score": row["Score"],
            })
    else:
        for idx, row in df_true.iterrows():
            output_rows.append({
                "Datetime": None,
                "Model": model_name,
                "WagonType": row.get("WagonType", ""),
                "Source": row.get("Source", ""),
                "Score": row["Score"],
            })

if output_rows:
    df_output = pd.DataFrame(output_rows)
    out_path = PROJECT_DIR / "output.csv"
    df_output.to_csv(out_path, index=False)
    print(f"Wrote {len(df_output)} prediction=True row(s) to {out_path}")
else:
    print("No predictions were True (or all filtered out); output.csv not written.")

df_output = df_output[df_output["Model"].str.contains("svm", case=False, na=False)]
df_output

In [ ]:
# Timetable plot of df_output (when Prediction is True)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if "df_output" in globals() and len(df_output) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    df_plot = df_output.copy()
    # Use Datetime if available, else Date
    if "Datetime" in df_plot.columns and df_plot["Datetime"].notna().any():
        df_plot["_time"] = pd.to_datetime(df_plot["Datetime"], errors="coerce")
    elif "Date" in df_plot.columns and df_plot["Date"].notna().any():
        df_plot["_time"] = pd.to_datetime(df_plot["Date"], errors="coerce")
    else:
        df_plot["_time"] = df_plot.index
    df_plot = df_plot.dropna(subset=["_time"])
    if len(df_plot) == 0:
        print("No valid dates to plot.")
    else:
        kit = df_plot["Source"].unique()
        y_map = {m: i for i, m in enumerate(kit)}
        df_plot["_y"] = df_plot["Source"].map(y_map)
        for model in kit:
            mask = df_plot["Source"] == model
            sub = df_plot.loc[mask]
            ax.scatter(sub["_time"], sub["_y"], label=model, alpha=0.8, s=50)
        ax.set_yticks(range(len(kit)))
        ax.set_yticklabels(kit, fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.xticks(rotation=45, ha="right")
        ax.set_xlabel("Datetime")
        ax.set_ylabel("Source")
        ax.set_title("Timetable: prediction=True events")
        ax.legend(loc="upper left", fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No prediction=True events to plot. Run the output cell above first.")

In [ ]:
summary_rows = []

for model_name, df_res in all_results.items():
    fp = (df_res["Prediction"] == 1).sum()
    n = len(df_res)

    summary_rows.append({
        "Model": model_name,
        "FalsePositives": fp,
        "TotalSamples": n,
        "FalseAlarmRate_%": 100 * fp / n if n > 0 else np.nan
    })

summary_df = pd.DataFrame(summary_rows).sort_values("FalseAlarmRate_%")
summary_df


In [ ]:
far_rows = []

for model_name, df_res in all_results.items():
    if "WagonType" not in df_res.columns:
        raise ValueError("Column 'WagonType' not found")

    for wagon, df_w in df_res.groupby("WagonType"):
        y_pred = df_w["Prediction"].astype(int).values

        fp = np.sum(y_pred == 1)
        tn = np.sum(y_pred == 0)
        n = len(y_pred)

        far_rows.append({
            "Model": model_name,
            "WagonType": wagon,
            "TN": tn,
            "FP": fp,
            "TotalSamples": n,
            "FalseAlarmRate_%": 100 * fp / n if n > 0 else np.nan
        })

far_df = pd.DataFrame(far_rows).sort_values(
    ["WagonType", "FalseAlarmRate_%"]
)

far_df


# Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrices = {}

for model_name, df_res in all_results.items():
    model_cm = {}

    for wagon, df_w in df_res.groupby("WagonType"):
        y_true = np.zeros(len(df_w), dtype=int)
        y_pred = df_w["Prediction"].astype(int).values

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

        model_cm[wagon] = cm

    confusion_matrices[model_name] = model_cm


In [ ]:
for model, wagon_dict in confusion_matrices.items():
    print(f"\nModel: {model}")
    for wagon, cm in wagon_dict.items():
        print(f"  WagonType: {wagon}")
        print(cm)


# Plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import math
from matplotlib.colors import LinearSegmentedColormap

def short_model_name(model_key: str) -> str:
    """
    Convert joblib stem or dict key into a compact model name
    """
    key = model_key.lower()

    if "rf" in key:
        return "RF"
    if "dt" in key:
        return "dt"
    if "knn" in key:
        return "KNN"
    if "svm" in key:
        return "SVM"

    # fallback (safe)
    return model_key


# ------------------------------------------------------------------
# Colormap (unchanged)
# ------------------------------------------------------------------
light_blues = LinearSegmentedColormap.from_list(
    "light_blues",
    [(0, "#eff1f3"), (0.1, "#8cc6fc"), (1, "#95d4f6")]
)


def plot_false_alarm_grid_by_wagon(
    all_results,
    wagon_col="WagonType",
    max_cols=4,
    exclude_models=None
):
    """
    For each WagonType: one figure with subplots for all models.
    Each subplot is a 1x2 confusion matrix [TN, FP],
    assuming all samples are Healthy (y_true = 0).
    """

    # --------------------------------------------------------------
    # Collect all wagon types across all models
    # --------------------------------------------------------------
    wagons = sorted({
        w
        for df in all_results.values()
        if wagon_col in df.columns
        for w in df[wagon_col].dropna().unique()
    })

    if len(wagons) == 0:
        raise ValueError(f"No wagons found using column '{wagon_col}'")

    exclude_models = exclude_models or []

    model_names = [
        m for m in all_results.keys()
        if short_model_name(m) not in exclude_models
    ]

    # --------------------------------------------------------------
    # One figure per wagon
    # --------------------------------------------------------------
    for w in wagons:
        n_models = len(model_names)
        ncols = min(max_cols, n_models)
        nrows = math.ceil(n_models / ncols)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(4.3 * ncols, 2.8 * nrows),
            constrained_layout=True
        )
        axes = np.atleast_1d(axes).reshape(nrows, ncols)

        # ----------------------------------------------------------
        # Compute confusion stats first (for consistent color scale)
        # ----------------------------------------------------------
        vmax = 1
        stats = {}

        for model in model_names:
            df_m = all_results[model]
            df_w = df_m[df_m[wagon_col] == w]

            if df_w.empty:
                stats[model] = None
                continue

            y_pred = df_w["Prediction"].astype(int).values
            y_true = np.zeros(len(y_pred), dtype=int)

            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            TN, FP = cm[0, 0], cm[0, 1]

            vmax = max(vmax, TN, FP)
            stats[model] = (TN, FP, len(y_pred))

        # ----------------------------------------------------------
        # Plot each model
        # ----------------------------------------------------------
        for idx, model in enumerate(model_names):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]

            if stats[model] is None:
                ax.axis("off")
                ax.set_title(f"{model}\n(no samples)", fontsize=10)
                continue

            TN, FP, total = stats[model]
            far = FP / total if total > 0 else 0.0

            im = ax.imshow(
                [[TN, FP]],
                cmap=light_blues,
                vmin=0,
                vmax=vmax,
                aspect="auto"
            )

            ax.text(0, 0, f"{TN}\n(TN)", ha="center", va="center",
                    fontsize=12, fontweight="bold")
            ax.text(1, 0, f"{FP}\n(FP)", ha="center", va="center",
                    fontsize=12, fontweight="bold")

            ax.set_xticks([0, 1])
            ax.set_xticklabels(["Healthy", "Combined\nLeakage"])
            ax.set_yticks([0])
            ax.set_yticklabels(["True\n Healthy"])

            label = short_model_name(model)

            ax.set_title(
                f"{label}",
                # f"{label}\nFAR={far:.2%} ({FP}/{total})",
                fontsize=16,
                fontweight="bold"
            )


            ax.grid(False)

        # ----------------------------------------------------------
        # Hide unused axes
        # ----------------------------------------------------------
        for j in range(n_models, nrows * ncols):
            r, c = divmod(j, ncols)
            axes[r, c].axis("off")

        # ----------------------------------------------------------
        # Figure-level annotations
        # ----------------------------------------------------------
        fig.suptitle(
            f"WagonType {w} — False Alarm Analysis",
            fontsize=20,
            fontweight="bold"
        )
        fig.supxlabel("Prediction", fontsize=16)
        fig.supylabel("Healthy-only", fontsize=16)

        # Shared colorbar
        # cbar = fig.colorbar(
        #     im,
        #     ax=axes.ravel().tolist(),
        #     shrink=0.85,
        #     pad=0.02
        # )
        # cbar.set_label("Sample count")

        plt.show()


In [ ]:
plt.rcParams.update({
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

In [ ]:
plot_false_alarm_grid_by_wagon(
    all_results,
    wagon_col="WagonType",
    exclude_models=["dt_tuned"]
)

In [ ]:
def plot_false_alarm_rf_by_wagon_binary(
    all_results,
    rf_model_key=None,
    wagon_col="WagonType",
    max_cols=4,
    labels=(0, 1),
    xticklabels=("Healthy", "Leakage"),
    yticklabel="True\n Healthy"
):
    """
    Plot only the Random Forest healthy-only false alarm results,
    with one subplot per WagonType.
    """

    rf_candidates = [m for m in all_results.keys() if short_model_name(m) == "RF"]

    if rf_model_key is None:
        if not rf_candidates:
            raise ValueError("No RF model found in all_results")
        rf_model_key = rf_candidates[0]
    elif rf_model_key not in all_results:
        raise KeyError(f"Model '{rf_model_key}' not found in all_results")

    df_rf = all_results[rf_model_key]

    if wagon_col not in df_rf.columns:
        raise ValueError(f"Column '{wagon_col}' not found in RF results")

    wagons = sorted(df_rf[wagon_col].dropna().unique())

    if len(wagons) == 0:
        raise ValueError(f"No wagons found using column '{wagon_col}'")

    n_wagons = len(wagons)
    ncols = min(max_cols, n_wagons)
    nrows = math.ceil(n_wagons / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5 * ncols, 3.5 * nrows),
        constrained_layout=True
    )
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    vmax = 1
    stats = {}

    for w in wagons:
        df_w = df_rf[df_rf[wagon_col] == w]

        if df_w.empty:
            stats[w] = None
            continue

        y_pred = df_w["Prediction"].astype(int).values
        y_true = np.zeros(len(y_pred), dtype=int)
        cm = confusion_matrix(y_true, y_pred, labels=list(labels))

        pred0 = cm[0, 0]
        pred1 = cm[0, 1]

        vmax = max(vmax, pred0, pred1)
        stats[w] = (pred0, pred1, len(y_pred))

    for idx, w in enumerate(wagons):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]

        if stats[w] is None:
            ax.axis("off")
            ax.set_title(f"WagonType {w}\n(no samples)", fontsize=10)
            continue

        pred0, pred1, total = stats[w]
        fp_total = pred1
        far_total = fp_total / total if total > 0 else 0.0

        ax.imshow(
            [[pred0, pred1]],
            cmap=light_blues,
            vmin=0,
            vmax=vmax,
            aspect="auto"
        )

        ax.text(0, 0, f"{pred0}\n(TN)", ha="center", va="center",
                fontsize=16, fontweight="bold")
        ax.text(1, 0, f"{pred1}\n(FP)", ha="center", va="center",
                fontsize=16, fontweight="bold")

        ax.set_xticks([0, 1])
        ax.set_xticklabels(list(xticklabels))
        ax.set_yticks([0])
        ax.set_yticklabels([yticklabel])

        ax.set_title(
            # f"WagonType {w}\nFAR={far_total:.2%} (FP={fp_total}/{total})",
            f"WagonType {w}",
            fontsize=16,
            fontweight="bold"
        )
        ax.grid(False)

    for j in range(n_wagons, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    fig.suptitle(
        f"RF Confusion Matrix for each Wagon (binary)",
        fontsize=20,
        fontweight="bold"
    )
    # fig.supxlabel("Prediction", fontsize=22)
    plt.show()


In [ ]:
plot_false_alarm_rf_by_wagon_binary(
    {
        model_name: df[df["WagonType"].astype(str).isin(["4909", "4575"])]
        for model_name, df in all_results.items()
    },
    wagon_col="WagonType",
    max_cols=4
)


# The basic

In [ ]:
# def get_base_dir():
#     try:
#         # .py execution
#         return Path(__file__).resolve().parent
#     except NameError:
#         # Jupyter / interactive
#         return Path.cwd()

# SCRIPT_DIR = get_base_dir()

# # Go ONE level up, then into saved_models
# PROJECT_DIR = SCRIPT_DIR

# MODEL_PATH = PROJECT_DIR / "model" / "20260120_231059_feat2_rf_tuned_inference_pipeline.joblib"

# bundle = joblib.load(MODEL_PATH)

# pipe = bundle["pipeline"]
# features = bundle["features"]

# test_data_path = ["TestBrakefinal_data_raw_Dati30.csv",
#                   "TestBrakefinal_data_raw_Dati05.csv",
#                   "TestBrakefinal_data_raw_Dati10.csv",
#                   "TestBrakefinal_data_raw_Dati11.csv",
#                   "TestBrakefinal_data_raw_Dati18.csv",
#                   "TestBrakefinal_data_raw_Dati24.csv"
#                   ]

# df_test = load_and_prepare_monorail(test_data_path)

# df_wv1 = df_test.loc[
#     (df_test['Non_Standard_Braking'].eq(0)) &
#     (df_test['BC_BadStart'].eq(0)) &
#     (df_test['WV_bin'].eq(1))
# ].copy()
# print(f"Filtered test data to {len(df_wv1)} samples with WV_bin=1 and standard braking.")
# X_new = df_wv1[features]
# y_pred = pipe.predict(X_new)

# y_true = np.zeros(len(y_pred), dtype=int)

# # Attach predictions to the original dataframe
# df_wv1 = df_wv1.copy()
# y_score = pipe.predict_proba(X_new)[:, 1]

# df_wv1["Score"] = y_score
# df_wv1["Prediction"] = y_pred

In [ ]:
# tn = np.sum(y_pred == 0)
# fp = np.sum(y_pred == 1)

# confusion_healthy = np.array([[tn, fp]])

# print("Confusion matrix (Healthy-only):")
# print(confusion_healthy)


In [ ]:
# far = fp / (fp + tn) if (fp + tn) > 0 else np.nan

# print(f"Total samples : {len(y_pred)}")
# print(f"False alarms  : {fp}")
# print(f"True negatives: {tn}")
# print(f"FAR           : {far:.4f}")


In [ ]:
# import pandas as pd

# results_df = pd.DataFrame(
#     {
#         "Predicted_Healthy (TN)": [tn],
#         "Predicted_Fault (FP)": [fp],
#         "Total": [tn + fp],
#         "False_Alarm_Rate": [far],
#     },
#     index=["Healthy (assumed)"]
# )

# results_df


In [ ]:
# if hasattr(pipe, "predict_proba"):
#     y_proba = pipe.predict_proba(X_new)[:, 1]

#     threshold = 0.9  # example
#     y_pred_thr = (y_proba >= threshold).astype(int)

#     tn_thr = np.sum(y_pred_thr == 0)
#     fp_thr = np.sum(y_pred_thr == 1)
#     far_thr = fp_thr / (fp_thr + tn_thr)

#     print(f"\nThreshold = {threshold}")
#     print(f"FP = {fp_thr}, TN = {tn_thr}, FAR = {far_thr:.4f}")
